<a href="https://colab.research.google.com/github/rosnaaidiploma-web/Rosnaelizabeth/blob/main/assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os  #Interact with the operating system
import random  #Generate random values
import numpy as np  #Mathematical computations
import matplotlib.pyplot as plt  #Data visualization
import tensorflow as tf  #Build neural networks
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array # image preprocessing #load image # convert image into array


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive # connect to drive
drive.mount('/content/drive')

In [ ]:
import tensorflow as tf #GPU Available
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True) # mounted drive


In [ ]:
data_paths = "/content/drive/MyDrive/myfolderros/Vehicles" # Path of original dataset

In [ ]:
data_paths # print the paths of original dataset

In [ ]:

import shutil #to bulit_in python library( copy,move, delete and manage directory operation )


source_folder = "/content/drive/MyDrive/myfolderros/Vehicles/Cars"
train_folder = "/content/Vehicles/train/Car"
val_folder = "/content/Vehicles/validation/Car"

# Create target directories if they don't exist
os.makedirs(train_folder, exist_ok=True)
os.makedirs(val_folder, exist_ok=True)

In [ ]:

classes = ["Auto Rickshaws", "Bikes", "Cars", "Motorcycles","Planes","Ships","Trains"]

# Use the correct root source path for all classes, which is `data_paths`
source_data_root = data_paths

# Define the root directories for the new train and validation split output
output_base_dir = "/content/Vehicles_split" # A new base directory to store the split data
output_train_root = os.path.join(output_base_dir, "train")
output_val_root = os.path.join(output_base_dir, "validation")

for class_name in classes:
    # Construct the path to the original class images (e.g., /Vehicles/Auto Rickshaws)
    class_source_path = os.path.join(source_data_root, class_name)

    # Check if the class directory exists before trying to list its contents
    if not os.path.exists(class_source_path):
        print(f"Warning: Source directory for class '{class_name}' not found: {class_source_path}. Skipping this class.")
        continue

    images = os.listdir(class_source_path)
    random.shuffle(images)

    split = int(0.8 * len(images))
    train_images = images[:split]
    val_images = images[split:]

    # Construct the target directories for this specific class in the split dataset
    # (e.g., /Vehicles_split/train/Auto Rickshaws)
    target_train_class_dir = os.path.join(output_train_root, class_name)
    target_val_class_dir = os.path.join(output_val_root, class_name)

    # Create target directories if they don't exist. os.makedirs handles parent directories.
    os.makedirs(target_train_class_dir, exist_ok=True)
    os.makedirs(target_val_class_dir, exist_ok=True)

    for img in train_images:
        shutil.copy(os.path.join(class_source_path, img),
                    os.path.join(target_train_class_dir, img))

    for img in val_images:
        shutil.copy(os.path.join(class_source_path, img),
                    os.path.join(target_val_class_dir, img))

print("Dataset splitting complete.")

In [ ]:
output_base_dir

In [ ]:
target_train_class_dir

In [ ]:
target_val_class_dir

In [ ]:
from tensorflow.keras.applications.vgg16 import preprocess_input

# Define train_dir and val_dir using the paths created earlier
train_dir = output_train_root
val_dir = output_val_root

# Training Data Generator (with small augmentation)
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

# Validation Data Generator (only preprocessing)
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)


# ==============================
# Load Images from Folder
# ==============================

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),   # Required size for VGG16
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)


# ==============================
# Print Information
# ==============================

print("Class Labels:", train_generator.class_indices)
print("Number of Training Images:", train_generator.samples)
print("Number of Validation Images:", val_generator.samples)

In [ ]:
from tensorflow.keras.applications import VGG16

In [ ]:
from tensorflow.keras.applications import VGG16

# Check if we want pretrained weights
use_pretrained = True   # Change to False if you don't want pretrained weights

if use_pretrained:
    print("Loading Pretrained VGG16 Model...")

    base_model = VGG16(
        weights='imagenet',      # Load ImageNet weights
        include_top=False,
        input_shape=(224, 224, 3)
    )
else:
    print("Loading VGG16 Model without pretrained weights...")

    base_model = VGG16(
        weights=None,            # No pretrained weights
        include_top=False,
        input_shape=(224, 224, 3)
    )

# Freeze layers
for layer in base_model.layers:
    layer.trainable = False

print("Model Loaded Successfully!")


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout


In [ ]:
def build_classifier(base_model, num_classes):

    model = Sequential()

    # Add base model (VGG16 without top)
    model.add(base_model)

    # Add Flatten layer
    model.add(Flatten())

    # Add Dense layers using for loop
    units = [256, 128]   # Number of neurons

    for u in units:
        model.add(Dense(u, activation='relu'))
        model.add(Dropout(0.5))

    # Final Output Layer
    model.add(Dense(num_classes, activation='softmax'))

    return model


In [ ]:
num_classes = 4   # Change according to your dataset

model = build_classifier(base_model, num_classes)

model.summary()


In [ ]:
from tensorflow.keras.optimizers import Adam

# Compile Function
def compile_model(model):

    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    print("Model Compiled Successfully!")

# Train Function
def train_model(model, train_generator, val_generator, epochs):

    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=epochs
    )

    return history

# Run
# Correct num_classes based on the generator
correct_num_classes = len(train_generator.class_indices)
print(f"Correcting num_classes from {num_classes} to {correct_num_classes}")

# Rebuild the model with the correct number of classes
# build_classifier and base_model are defined in previous cells and should be in scope.
model = build_classifier(base_model, correct_num_classes)

compile_model(model)
history = train_model(model, train_generator, val_generator, 5)

In [ ]:
def plot_graph(history, choice):

    if choice == 1:
        # Plot Accuracy Graph
        plt.plot(history.history['accuracy'])
        plt.plot(history.history['val_accuracy'])
        plt.title("Model Accuracy")
        plt.xlabel("Epochs")
        plt.ylabel("Accuracy")
        plt.legend(["Train Accuracy", "Validation Accuracy"])
        plt.show()

    elif choice == 2:
        # Plot Loss Graph
        plt.plot(history.history['loss'])
        plt.plot(history.history['val_loss'])
        plt.title("Model Loss")
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.legend(["Train Loss", "Validation Loss"])
        plt.show()

    elif choice == 3:
        # Plot Both Graphs One by One
        print("Showing Accuracy Graph")
        plt.plot(history.history['accuracy'])
        plt.plot(history.history['val_accuracy'])
        plt.title("Model Accuracy")
        plt.xlabel("Epochs")
        plt.ylabel("Accuracy")
        plt.legend(["Train", "Validation"])
        plt.show()

        print("Showing Loss Graph")
        plt.plot(history.history['loss'])
        plt.plot(history.history['val_loss'])
        plt.title("Model Loss")
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.legend(["Train", "Validation"])
        plt.show()

    else:
        print("Invalid Choice! Select 1, 2 or 3")


In [ ]:
# 1 → Accuracy
# 2 → Loss
# 3 → Both

plot_graph(history, 1)


In [ ]:
model.save("vehicle_classifier_vgg16.keras")
print("Model saved successfully!")


In [ ]:
from google.colab import files
files.download("vehicle_classifier_vgg16.keras")


In [ ]:
def test_single_image(model, img_path):

    # Check if file exists
    if os.path.exists(img_path):

        print("Image Found! Testing...")

        # Load image
        img = image.load_img(img_path, target_size=(224, 224))
        img_array = image.img_to_array(img)

        # Expand dimensions
        img_array = np.expand_dims(img_array, axis=0)

        # Normalize (if you used preprocess_input, use that instead)
        img_array = img_array / 255.0

        # Predict
        prediction = model.predict(img_array)
        predicted_index = np.argmax(prediction)

        # Get class names
        class_names = list(train_generator.class_indices.keys())
        predicted_class = class_names[predicted_index]

        # Show image
        plt.imshow(img)
        plt.title("Predicted: " + predicted_class)
        plt.axis("off")
        plt.show()

        print("Predicted Class:", predicted_class)

    else:
        print("Image path not found!")


In [ ]:
def test_single_image(model, img_path):

    # Check if file exists
    if os.path.exists(img_path):

        print("Image Found! Testing...")

        # Load image
        img = load_img(img_path, target_size=(224, 224)) # Corrected: use load_img directly
        img_array = img_to_array(img) # Corrected: use img_to_array directly

        # Expand dimensions
        img_array = np.expand_dims(img_array, axis=0)

        # Normalize (if you used preprocess_input, use that instead)
        img_array = img_array / 255.0

        # Predict
        prediction = model.predict(img_array)
        predicted_index = np.argmax(prediction)

        # Get class names
        class_names = list(train_generator.class_indices.keys())
        predicted_class = class_names[predicted_index]

        # Show image
        plt.imshow(img)
        plt.title("Predicted: " + predicted_class)
        plt.axis("off")
        plt.show()

        print("Predicted Class:", predicted_class)

    else:
        print("Image path not found!")

image_path = "/content/drive/MyDrive/myfolderros/Vehicles/Planes/Plane (100).jpg"   # Change to your image path
test_single_image(model, image_path)

In [ ]:
def show_top3_predictions(model, img_path):

    if os.path.exists(img_path):

        print("Image Found! Predicting...")

        # Load Image
        img = image.load_img(img_path, target_size=(224, 224))
        img_array = image.img_to_array(img)

        # Expand dimension
        img_array = np.expand_dims(img_array, axis=0)

        # Normalize
        img_array = img_array / 255.0

        # Predict
        prediction = model.predict(img_array)

        # Get class names
        class_names = list(train_generator.class_indices.keys())

        # Convert to percentage
        probabilities = prediction[0] * 100

        # Get Top 3 indices
        top3_indices = np.argsort(probabilities)[-3:][::-1]

        # Get Top 3 class names and probabilities
        top3_classes = [class_names[i] for i in top3_indices]
        top3_probs = [probabilities[i] for i in top3_indices]

        # Show Image
        plt.imshow(img)
        plt.title("Test Image")
        plt.axis("off")
        plt.show()

        # Plot Top 3 Bar Graph
        plt.figure()
        plt.bar(top3_classes, top3_probs)
        plt.xlabel("Top 3 Classes")
        plt.ylabel("Probability (%)")
        plt.title("Top 3 Prediction Probabilities")
        plt.show()

        # Print Top 3 Results
        print("Top 3 Predictions:")
        for i in range(3):
            print(f"{i+1}. {top3_classes[i]} - {top3_probs[i]:.2f}%")

    else:
        print("Image Path Not Found!")


In [ ]:
def show_top3_predictions(model, img_path):

    if os.path.exists(img_path):

        print("Image Found! Predicting...")

        # Load Image
        img = load_img(img_path, target_size=(224, 224)) # Corrected: use load_img directly
        img_array = img_to_array(img) # Corrected: use img_to_array directly

        # Expand dimension
        img_array = np.expand_dims(img_array, axis=0)

        # Normalize
        img_array = img_array / 255.0

        # Predict
        prediction = model.predict(img_array)

        # Get class names
        class_names = list(train_generator.class_indices.keys())

        # Convert to percentage
        probabilities = prediction[0] * 100

        # Get Top 3 indices
        top3_indices = np.argsort(probabilities)[-3:][::-1]

        # Get Top 3 class names and probabilities
        top3_classes = [class_names[i] for i in top3_indices]
        top3_probs = [probabilities[i] for i in top3_indices]

        # Show Image
        plt.imshow(img)
        plt.title("Test Image")
        plt.axis("off")
        plt.show()

        # Plot Top 3 Bar Graph
        plt.figure()
        plt.bar(top3_classes, top3_probs)
        plt.xlabel("Top 3 Classes")
        plt.ylabel("Probability (%)")
        plt.title("Top 3 Prediction Probabilities")
        plt.show()

        # Print Top 3 Results
        print("Top 3 Predictions:")
        for i in range(3):
            print(f"{i+1}. {top3_classes[i]} - {top3_probs[i]:.2f}%")

    else:
        print("Image Path Not Found!")

image_path = "/content/drive/MyDrive/myfolderros/Vehicles/Planes/Plane (100).jpg"
show_top3_predictions(model, image_path)


In [ ]:
def predict_multiple_images(model, folder_path):

    if os.path.exists(folder_path):

        print("Folder Found! Starting Predictions...\n")

        # Get class names
        class_names = list(train_generator.class_indices.keys())

        # Loop through all images in folder
        for img_name in os.listdir(folder_path):

            img_path = os.path.join(folder_path, img_name)

            # Check only image files
            if img_name.endswith( (".jpg", ".jpeg", ".png") ):

                print("Predicting:", img_name)

                # Load image
                img = load_img(img_path, target_size=(224, 224))
                img_array = img_to_array(img)

                # Expand dimension
                img_array = np.expand_dims(img_array, axis=0)

                # Normalize
                img_array = img_array / 255.0

                # Predict
                prediction = model.predict(img_array)

                # Get predicted index
                predicted_index = np.argmax(prediction)

                # Get class name
                predicted_class = class_names[predicted_index]

                # Confidence
                confidence = prediction[0][predicted_index] * 100

                # Show Image
                plt.imshow(img)
                plt.title(f"{predicted_class} ({confidence:.2f}%)")
                plt.axis("off")
                plt.show()

                print("Predicted Class:", predicted_class)
                print("Confidence: {:.2f}%".format(confidence))
                print("-" * 40)

    else:
        print("Folder Path Not Found!")


In [ ]:
import tensorflow as tf # Import tensorflow to load the model

# Check if 'model' is defined, if not, load it
if 'model' not in locals() and 'model' not in globals():
    try:
        model = tf.keras.models.load_model('vehicle_classifier_vgg16.keras')
        print("Model loaded successfully from vehicle_classifier_vgg16.keras")
    except Exception as e:
        print(f"Error loading model: {e}. Please ensure previous cells defining and saving the model were run, or that the file exists.")

test_folder = "/content/drive/MyDrive/myfolderros/Vehicles/Planes"
predict_multiple_images(model, test_folder)


In [ ]:
def count_images_per_class(model, folder_path):

    if os.path.exists(folder_path):

        print("Folder Found! Counting Predictions...\n")

        # Get class names
        class_names = list(train_generator.class_indices.keys())

        # Create dictionary to store counts
        class_count = {}

        # Initialize count 0 for each class
        for class_name in class_names:
            class_count[class_name] = 0

        # Loop through images
        for img_name in os.listdir(folder_path):

            if img_name.endswith((".jpg", ".jpeg", ".png")):

                img_path = os.path.join(folder_path, img_name)

                # Load image
                img = image.load_img(img_path, target_size=(224, 224))
                img_array = image.img_to_array(img)

                # Expand dimension
                img_array = np.expand_dims(img_array, axis=0)

                # Normalize
                img_array = img_array / 255.0

                # Predict
                prediction = model.predict(img_array)

                # Get predicted class index
                predicted_index = np.argmax(prediction)

                # Get class name
                predicted_class = class_names[predicted_index]

                # Increase count
                class_count[predicted_class] += 1

        # Print Results
        print("Prediction Count Per Class:\n")
        for class_name, count in class_count.items():
            print(class_name, ":", count)

    else:
        print("Folder Path Not Found!")


In [ ]:
import os

file_name = "vehicle_classifier_vgg16.keras"
current_dir_path = os.path.join(os.getcwd(), file_name)
content_dir_path = os.path.join("/content/", file_name)

if os.path.exists(current_dir_path):
    print(f"File found at: {current_dir_path}")
elif os.path.exists(content_dir_path):
    print(f"File found at: {content_dir_path}")
else:
    print(f"File '{file_name}' not found in current directory or /content/.")

In [ ]:
import tensorflow as tf # Import tensorflow to load the model

# Check if 'model' is defined, if not, load it
if 'model' not in locals() and 'model' not in globals():
    try:
        model = tf.keras.models.load_model('vehicle_classifier_vgg16.keras')
        print("Model loaded successfully from vehicle_classifier_vgg16.keras")
    except Exception as e:
        print(f"Error loading model: {e}. Please ensure previous cells defining and saving the model were run, or that the file exists.")

test_folder = "/content/drive/MyDrive/myfolderros/Vehicles/Planes/Plane (100).jpg"
count_images_per_class(model, test_folder)

In [ ]:
import os

file_name = "vehicle_classifier_vgg16.keras"

print("Checking current directory:")
if file_name in os.listdir('.'):
    print(f"File '{file_name}' found in current directory ({os.getcwd()})")
else:
    print(f"File '{file_name}' not found in current directory.")

print("\nChecking /content/ directory:")
if os.path.exists('/content/') and file_name in os.listdir('/content/'):
    print(f"File '{file_name}' found in /content/ directory.")
else:
    print(f"File '{file_name}' not found in /content/ directory.")

print("\nListing files in current directory (first 10):")
print(os.listdir('.')[:10])

print("\nListing files in /content/ directory (first 10):")
print(os.listdir('/content/')[:10])

In [ ]:
def count_images_per_class(model, folder_path):

    if os.path.exists(folder_path):

        print("Folder Found! Counting Predictions...\n")

        # Get class names
        class_names = list(train_generator.class_indices.keys())

        # Create dictionary to store counts
        class_count = {}

        # Initialize count 0 for each class
        for class_name in class_names:
            class_count[class_name] = 0

        # Loop through images
        for img_name in os.listdir(folder_path):

            if img_name.endswith((".jpg", ".jpeg", ".png")):

                img_path = os.path.join(folder_path, img_name)

                # Load image
                img = image.load_img(img_path, target_size=(224, 224))
                img_array = image.img_to_array(img)

                # Expand dimension
                img_array = np.expand_dims(img_array, axis=0)

                # Normalize
                img_array = img_array / 255.0

                # Predict
                prediction = model.predict(img_array)

                # Get predicted class index
                predicted_index = np.argmax(prediction)

                # Get class name
                predicted_class = class_names[predicted_index]

                # Increase count
                class_count[predicted_class] += 1

        # Print Results
        print("Prediction Count Per Class:\n")
        for class_name, count in class_count.items():
            print(class_name, ":", count)

    else:
        print("Folder Path Not Found!")
